# 2. Data Preprocessing
Messy text columns (`ScreenResolution`, `Memory`) ko structured columns mein todna.

**Input:** `data/processed/01_ingested.csv`  
**Output:** `data/processed/02_preprocessed.csv`

In [1]:
import re
import pandas as pd
import numpy as np

from common import INGESTED_FILE, PREPROCESSED_FILE

In [2]:
df = pd.read_csv(INGESTED_FILE)
print(df.shape)
df.head()

(1274, 11)


,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price
0,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37,71378.6832
1,Apple,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34,47895.5232
2,HP,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8,256GB SSD,Intel HD Graphics 620,No OS,1.86,30636.0000
3,Apple,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16,512GB SSD,AMD Radeon Pro 455,macOS,1.83,135195.3360
4,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37,96095.8080


## Screen resolution
`ScreenResolution` mein touchscreen, IPS aur resolution (e.g. `1920x1080`) sab ek string mein hai.

In [3]:
df["TouchScreen"] = df["ScreenResolution"].str.contains("Touchscreen", case=False).astype(int)
df["IPS"] = df["ScreenResolution"].str.contains("IPS", case=False).astype(int)

res = df["ScreenResolution"].str.extract(r"(\d{3,4})\s*x\s*(\d{3,4})").astype(int)
df["X_res"], df["Y_res"] = res[0], res[1]

df[["ScreenResolution", "TouchScreen", "IPS", "X_res", "Y_res"]].head()

,ScreenResolution,TouchScreen,IPS,X_res,Y_res
0,IPS Panel Retina Display 2560x1600,0,1,2560,1600
1,1440x900,0,0,1440,900
2,Full HD 1920x1080,0,0,1920,1080
3,IPS Panel Retina Display 2880x1800,0,1,2880,1800
4,IPS Panel Retina Display 2560x1600,0,1,2560,1600


In [4]:
assert df[["X_res", "Y_res"]].notna().all().all()
df = df.drop(columns=["ScreenResolution"])
print("TouchScreen share:", round(df["TouchScreen"].mean(), 3), "| IPS share:", round(df["IPS"].mean(), 3))

TouchScreen share: 0.148 | IPS share: 0.28


## Memory
Examples: `256GB SSD`, `128GB SSD +  1TB HDD`, `1.0TB Hybrid`.  
Har laptop ke liye SSD / HDD / Hybrid / Flash Storage ki total size GB mein nikalte hain (1TB = 1000GB).

In [5]:
MEM_PATTERN = re.compile(r"(\d+(?:\.\d+)?)\s*(GB|TB)\s*(SSD|HDD|Hybrid|Flash Storage)")

def parse_memory(text):
    totals = {"HDD": 0, "SSD": 0, "Hybrid": 0, "Flash_Storage": 0}
    for size, unit, kind in MEM_PATTERN.findall(text):
        gb = float(size) * (1000 if unit == "TB" else 1)
        totals[kind.replace(" ", "_")] += int(gb)
    return pd.Series(totals)

mem = df["Memory"].apply(parse_memory)
mem.head()

,HDD,SSD,Hybrid,Flash_Storage
0,0,128,0,0
1,0,0,0,128
2,0,256,0,0
3,0,512,0,0
4,0,256,0,0


In [6]:
# Every row must have been understood by the parser
assert (mem.sum(axis=1) > 0).all(), "Some Memory values could not be parsed"

df = pd.concat([df.drop(columns=["Memory"]), mem], axis=1)
df[["HDD", "SSD", "Hybrid", "Flash_Storage"]].describe().T

,count,mean,std,min,25%,50%,75%,max
HDD,1274.0,413.390895,517.980670,0.0,0.0,0.0,1000.0,2000.0
SSD,1274.0,186.106750,186.531295,0.0,0.0,256.0,256.0,1024.0
Hybrid,1274.0,7.463108,84.952298,0.0,0.0,0.0,0.0,1000.0
Flash_Storage,1274.0,4.508634,30.500079,0.0,0.0,0.0,0.0,512.0


In [7]:
df.to_csv(PREPROCESSED_FILE, index=False)
print("Saved:", PREPROCESSED_FILE.name, df.shape)
df.head()

Saved: 02_preprocessed.csv (1274, 17)


,Company,TypeName,Inches,Cpu,Ram,Gpu,OpSys,Weight,Price,TouchScreen,IPS,X_res,Y_res,HDD,SSD,Hybrid,Flash_Storage
0,Apple,Ultrabook,13.3,Intel Core i5 2.3GHz,8,Intel Iris Plus Graphics 640,macOS,1.37,71378.6832,0,1,2560,1600,0,128,0,0
1,Apple,Ultrabook,13.3,Intel Core i5 1.8GHz,8,Intel HD Graphics 6000,macOS,1.34,47895.5232,0,0,1440,900,0,0,0,128
2,HP,Notebook,15.6,Intel Core i5 7200U 2.5GHz,8,Intel HD Graphics 620,No OS,1.86,30636.0000,0,0,1920,1080,0,256,0,0
3,Apple,Ultrabook,15.4,Intel Core i7 2.7GHz,16,AMD Radeon Pro 455,macOS,1.83,135195.3360,0,1,2880,1800,0,512,0,0
4,Apple,Ultrabook,13.3,Intel Core i5 3.1GHz,8,Intel Iris Plus Graphics 650,macOS,1.37,96095.8080,0,1,2560,1600,0,256,0,0
